# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to explore and process the FAIRˆ² mass spectrometry dataset using the `mlcroissant` library and its Croissant schema.

### Dataset Source
This dataset is described by a Croissant schema at the following URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load the dataset metadata using `mlcroissant`, then print the dataset's high-level metadata.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load via mlcroissant
dataset = mlc.Dataset(croissant_url)

# Print high-level dataset metadata
print(f'Name: {dataset.metadata.name}')
print(f'Description: {dataset.metadata.description}')
print(f"Published: {getattr(dataset.metadata, 'datePublished', 'N/A')}")
print(f"Version: {getattr(dataset.metadata, 'version', 'N/A')}")
print(f"Keywords: {getattr(dataset.metadata, 'keywords', [])}")

## 2. Data Overview
Review all available record sets, and identify their fields and columns by their `@id` values.

We list the record sets declared in the Croissant schema and print the available fields and columns, always referencing entities by their `@id`.

In [ ]:
# List all record sets and their details

record_sets = list(dataset.record_sets)
print(f"Available record sets (@id):");
for rs in record_sets:
    print(f"  - {rs.id} (name: {getattr(rs, 'name', '(no name)')})")
    print(f"    Fields:")
    for field in rs.fields:
        print(f"      * {field.id} (name: {getattr(field, 'name', '(no name)')})")
        # If the field is from tabular data, print column @id as well
        if hasattr(field, 'columns'):
            for col in field.columns:
                print(f"          - Column: {col.id} (name: {getattr(col, 'name', '')})")

## 3. Data Extraction
Let's extract the records of the main protein record set, and convert to a Pandas DataFrame for further analysis. In all code, we always reference entities by their Croissant `@id` field.

Replace `<your_main_record_set_id>` with the `@id` discovered above for the main tabular record set (e.g. something like `cr:RecordSet/1`).

In [ ]:
# Choose the main tabular record set (edit @id as needed based on output above)
main_record_set_id = None
for rs in dataset.record_sets:
    # Heuristics: try to select the largest record set, or the one with fields that look like protein data
    if hasattr(rs, 'fields') and len(rs.fields) > 5:
        main_record_set_id = rs.id
        break
if not main_record_set_id:
    main_record_set_id = record_sets[0].id  # fallback to the first

print(f"Selected record set: {main_record_set_id}")

# Load all records from this record set into a DataFrame
records = list(dataset.records(record_set=main_record_set_id))
df = pd.DataFrame(records)
print(f"Data columns: {df.columns.tolist()}")
df.head()

## 4. Exploratory Data Analysis (EDA)
Filter, normalize, and group data using only Croissant `@id`. Here, we:
- Pick a numeric field (e.g. `cr:Field/coverage_percent` or another numeric field `@id` from your field list).
- Filter for high values,
- Normalize,
- Optionally, group by a string field such as `cr:Field/sample_id` if present and numeric groupings are meaningful.

**Please change the variable assignments below according to the field `@id` you want to analyze, as listed above.**

In [ ]:
# Replace with the actual numeric field @id from data overview above
numeric_field_id = None
group_field_id = None
for col in df.columns:
    if 'coverage' in col or 'MW' in col or 'abundance' in col:
        numeric_field_id = col
        break
# Try to find a categorical grouping field
for col in df.columns:
    if 'sample' in col or 'condition' in col:
        group_field_id = col
        break
if numeric_field_id is None:
    numeric_field_id = df.columns[0]  # fallback
print(f"Operating on numeric field: {numeric_field_id}")
if group_field_id:
    print(f"Grouping field: {group_field_id}")

# Filtering, normalization, grouping
threshold = df[numeric_field_id].astype(float).mean()  # threshold = mean as example
filtered_df = df[df[numeric_field_id].astype(float) > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold} (mean):")
print(filtered_df[[numeric_field_id]].head())

# Normalization of the numeric field
filtered_df[numeric_field_id + "_normalized"] = (
    filtered_df[numeric_field_id].astype(float) - filtered_df[numeric_field_id].astype(float).mean()
) / filtered_df[numeric_field_id].astype(float).std()
print(filtered_df[[numeric_field_id, numeric_field_id + "_normalized"]].head())

# Optional: Group by another field (e.g., by sample, if present)
if group_field_id and group_field_id in df.columns:
    grouped_df = (
        filtered_df
        .groupby(group_field_id)[numeric_field_id]
        .mean()
        .reset_index()
    )
    print(f"Mean {numeric_field_id} grouped by {group_field_id}:")
    print(grouped_df.head())

## 5. Visualization
Visualize distributions of numeric values and relationships to groupings using matplotlib. All visualizations reference field variables by their Croissant `@id`.

Example: distribution of the selected numeric field, and barplot of group means if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8,4))
sns.histplot(df[numeric_field_id].astype(float), bins=30, kde=True)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel("Count")
plt.show()

if group_field_id is not None and group_field_id in df.columns:
    plt.figure(figsize=(10,4))
    means = (
        df.groupby(group_field_id)[numeric_field_id]
        .mean()
        .sort_values(ascending=False)
    )
    sns.barplot(x=means.index, y=means.values)
    plt.xticks(rotation=45, ha='right')
    plt.title(f"Mean {numeric_field_id} by {group_field_id}")
    plt.ylabel(numeric_field_id)
    plt.xlabel(group_field_id)
    plt.tight_layout()
    plt.show()

## 6. Conclusion
- Using `mlcroissant` and the dataset's Croissant schema, we explored the structure, loaded tabular data using record set and field `@id` values, and performed basic filtering, normalization and visualization.
- All references to record sets and fields were via their Croissant `@id`, ensuring reproducibility even if label names change.

You can now proceed with further FAIR data analyses and custom domain-specific tasks as needed.